# 3. Turn predictions into a contact list

**Question:** Given limited contact capacity, which customers would we consider contacting?

Run notebooks 01 and 02 first. This notebook loads the saved model in a separate session.
The default input is the full test table for an offline demonstration, not a live customer feed.

In [ ]:
import sys
from pathlib import Path

# Jupyter may start in the project root or in notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "source.json").exists():
    raise FileNotFoundError("Open Jupyter from the RetainAI project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUTPUT = ROOT / "output"
OUTPUT.mkdir(exist_ok=True)

## Load the model and choose a batch

For new data, put a separate CSV in `data/` and change INPUT_PATH below. Copy the columns from
`output/sample_customers.csv`. The 12 predictors, unique customer_id, and customer_value are required;
churn is not required. Never overwrite the original source CSV. New data files are ignored by default;
keep private customer information local.

In [ ]:
import pandas as pd

from retainai.scoring import load_model, score_customers

INPUT_PATH = OUTPUT / "test.csv"
bundle = load_model()
customers = pd.read_csv(INPUT_PATH, dtype={"customer_id": str})
print("Loaded model:", bundle["metadata"]["selected_model"])
customers.head()

## Set explicit scenario assumptions

These numbers are examples, not learned treatment effects or a recommended business budget.
Customer Value has no documented currency, so COST must be interpreted in the same hypothetical units.
RETENTION_RATE is the assumed fraction of at-risk value saved by contact, not model accuracy.

For probability 0.8 and value 1,000: expected loss = 800; assumed net benefit = 800 × 0.25 − 20 = 180.
There are no campaign outcomes in this dataset to establish that contacting this person saves 180.

In [ ]:
CAPACITY = 100  # Maximum number of contacts in this batch.
COST = 20.0  # Assumed cost for each contact, in source value units.
RETENTION_RATE = 0.25  # Assumed fraction of at-risk value saved.

scores = score_customers(
    customers, bundle, capacity=CAPACITY, cost=COST, retention_rate=RETENTION_RATE
)
scores = scores.sort_values("priority_rank")
scores.head(10)

## Read the scoring rules

The short `score_customers` function in `retainai/scoring.py` is the single implementation of these rules:

1. Validate the customer inputs.
2. Predict the probability of churn.
3. Multiply probability by customer value to get expected churn loss.
4. Multiply that loss by the assumed retention rate, then subtract contact cost.
5. Rank by expected loss, highest first. Ties follow input order.
6. Recommend contact only within capacity and when the unrounded scenario benefit is positive.

The displayed 0/1 prediction uses 0.5. Risk tiers use 0.3 and 0.7. These are illustrative labels,
not tuned business thresholds; the contact rule uses expected loss and benefit instead.
Constant cost and retention rate make ranking by loss equivalent to ranking by scenario benefit.

In [ ]:
recommended = scores[scores["contact_recommended"]]
assert len(recommended) <= CAPACITY
assert (recommended["priority_rank"] <= CAPACITY).all()
assert (
    recommended["scenario_net_benefit"] >= 0
).all()  # Display rounding can produce zero.
scores.to_csv(OUTPUT / "scores.csv", index=False)
print("Scored customers:", len(scores))
print("Suggested contacts:", len(recommended))
print("Assumed contact cost:", len(recommended) * COST)
print(
    "Assumed net benefit (rounded rows):",
    round(recommended.scenario_net_benefit.sum(), 2),
)
print("Saved output/scores.csv")

## Try one controlled change

Set CAPACITY to 0 and rerun the scenario and save cells: no contacts should be recommended.
Restore 100 afterward. Increase COST and see why fewer customers qualify. Changing these settings
does not retrain the model or change churn probabilities. Next: `04_drift.ipynb`.